# Course 01 — Why Agentic Coding Changes the PDLC

**Scenario.** Northstar Mutual wants to add policy-document question answering to its Underwriter Assistant. An implementation agent receives a small feature request, but the real change is constrained by organization policy, platform standards, insurance-domain rules, project architecture, and feature behavior.

**Success criteria.** Compose those layers without silent overrides; compare prompt-only and bounded proposals; inspect traces and metrics; route changes proportionately; inject a policy conflict; run real candidate code in a temporary repository; and explain what the evidence still cannot prove.

**Safety boundary.** Everything is deterministic and offline. Lab A simulates the control plane. Lab B copies a local training fixture to a temporary workspace and runs its tests; it does not contact a model, identity system, cloud service, or production repository. String permissions and training approvals are teaching data—not enforcement.

## Learning objectives and prerequisites

You will model specifications as an agent control plane, distinguish decision authority from execution authority, evaluate applicability and provenance, run actual candidate code through independent gates, inspect evidence gaps, and choose between direct change, lightweight specification, full SDD, and specialist-reviewed SDD. Basic Python and pull-request familiarity are sufficient.

## Environment and reproducibility

The notebook uses Python 3.10+ standard-library features and fixed fixtures. Run it from this lesson directory or the repository root. The next cell imports the reusable lab with `runpy`; the notebook does not maintain a second implementation.

In [ ]:
from dataclasses import replace
from pathlib import Path
import runpy

candidates = [
    Path('lab.py'),
    Path('curriculum/beginner/01-why-agentic-coding-changes-pdlc/lab.py'),
]
lab_path = next((path for path in candidates if path.exists()), None)
assert lab_path is not None, 'Run from the lesson directory or repository root.'
lab = runpy.run_path(str(lab_path))

Layer = lab['Layer']
GateDecision = lab['GateDecision']
Workflow = lab['Workflow']
Requirement = lab['Requirement']
ProposedDecision = lab['ProposedDecision']
Task = lab['Task']
AgentProposal = lab['AgentProposal']
ChangeProfile = lab['ChangeProfile']
compose_context = lab['compose_context']
evaluate_proposal = lab['evaluate_proposal']
recommend_workflow = lab['recommend_workflow']
enterprise_requirements = lab['enterprise_requirements']
prompt_only_proposal = lab['prompt_only_proposal']
controlled_proposal = lab['controlled_proposal']
default_boundary = lab['default_boundary']
inject_feature_policy_override = lab['inject_feature_policy_override']
print(f'Loaded reusable lab: {lab_path}')

## Architecture: inherited context constrains a feedback loop

The upper flow resolves organization, platform, domain, project, and feature requirements. The lower flow turns intent into a specification, plan, work graph, bounded agent execution, evidence, approval, deployment, and runtime feedback. The agent is one component—not the owner of the entire lifecycle.

![Enterprise Agentic PDLC control loop showing inherited specification context above a bounded delivery and evidence loop](assets/agentic-pdlc-control-loop.svg)

## Baseline context: inspect the hierarchy before execution

Each requirement has a stable ID, authority layer, control key, expected value, and statement. This simplified representation lets us observe precedence and provenance. Real requirements also need owner, applicability, version, status, rationale, and exception metadata.

In [ ]:
requirements = enterprise_requirements()
for requirement in requirements:
    print(f'{requirement.layer.name:12} {requirement.requirement_id:5} {requirement.control:24} = {requirement.expected_value}')

assert len(requirements) == 14
assert {requirement.layer for requirement in requirements} == {
    Layer.ORGANIZATION, Layer.PLATFORM, Layer.DOMAIN, Layer.PROJECT, Layer.FEATURE
}

## Compose effective context

Composition processes higher-authority layers first. Compatible duplicate controls preserve every source ID; incompatible lower-layer values become conflicts rather than overrides. The initial fixture should have no conflict.

In [ ]:
context = compose_context(requirements)
print('effective controls:', len(context.controls))
print('source requirement IDs:', len(context.requirement_ids))
print('conflicts:', context.conflicts)
assert len(context.controls) == 14
assert len(context.requirement_ids) == 14
assert context.conflicts == ()

## Experiment 1 — Prompt-only execution

The baseline proposal responds to “add policy-document Q&A” without the effective context. We evaluate it against the actual Northstar controls and an execution boundary: only the policy-assistant repository, feature-branch writes, test execution, at most 20 changed files, and human approval.

Observe categories rather than only the final STOP: missing controls, contradictory decisions, a retention decision made above implementation authority, unauthorized repository and permissions, excess change size, invalid task links, and weak evidence coverage.

In [ ]:
boundary = default_boundary()
baseline_report = evaluate_proposal(context, prompt_only_proposal(), boundary)
print('gate:', baseline_report.decision.value)
print('requirement coverage:', baseline_report.requirement_coverage)
print('evidence coverage:', baseline_report.evidence_coverage)
print('task traceability:', baseline_report.task_traceability)
print('violations:', *baseline_report.violations, sep='\n  - ')
print('overreach:', *baseline_report.autonomy_overreach, sep='\n  - ')
print('boundary:', *baseline_report.boundary_violations, sep='\n  - ')
assert baseline_report.decision is GateDecision.STOP
assert baseline_report.requirement_coverage < 0.1
assert baseline_report.autonomy_overreach

The evaluator does not claim the agent would actually choose these technologies. The fixture isolates a systemic risk: if a decision is required but absent from context, autonomous execution either blocks or fills the gap. A fast, internally coherent implementation can still be wrong for the organization.

## Incremental control: layered proposal and approval

The controlled proposal addresses the effective controls, gives every task valid requirement links, supplies evidence references, stays within repository/permission/file budgets, and makes only a local L5 structure choice without an owning requirement. We first evaluate without the required human approval, then grant it.

In [ ]:
proposal = controlled_proposal(context)
before_approval = evaluate_proposal(context, proposal, boundary)
after_approval = evaluate_proposal(context, proposal, boundary, human_approval_granted=True)

print('before approval:', before_approval.decision.value, before_approval.trace[-2:])
print('after approval:', after_approval.decision.value, after_approval.trace[-2:])
print('metrics:', after_approval.requirement_coverage, after_approval.evidence_coverage, after_approval.task_traceability)
assert before_approval.decision is GateDecision.REVIEW
assert after_approval.decision is GateDecision.PASS
assert (after_approval.requirement_coverage, after_approval.evidence_coverage, after_approval.task_traceability) == (1.0, 1.0, 1.0)

The metrics are deliberately narrow. A score of 1.0 means each represented control has a matching proposed value, each requirement ID appears in the evidence list, and each task links only to known requirements. It does not validate the requirement statements, architecture, tests, evidence quality, user value, or production behavior.

## Experiment 2 — Context ablation

What happens when the evaluator receives only project and feature requirements? We construct a proposal that perfectly conforms to that reduced context. A clean result can indicate missing governance context rather than a safe change.

In [ ]:
narrow_requirements = tuple(r for r in requirements if r.layer in {Layer.PROJECT, Layer.FEATURE})
narrow_context = compose_context(narrow_requirements)
narrow_decisions = tuple(
    ProposedDecision(c.control, c.expected_value, c.authority_layer, 'Visible in reduced context')
    for c in narrow_context.controls
)
narrow_proposal = AgentProposal(
    name='project-and-feature-only proposal',
    decisions=narrow_decisions,
    tasks=(Task('T-LOCAL', 'Implement the visible contract.', narrow_context.requirement_ids),),
    evidence_requirement_ids=narrow_context.requirement_ids,
    target_repositories=('policy-assistant',),
    requested_permissions=('read_source', 'write_feature_branch', 'run_tests'),
    estimated_files_changed=8,
)
narrow_report = evaluate_proposal(narrow_context, narrow_proposal, boundary, human_approval_granted=True)
print('reduced controls:', len(narrow_context.controls), 'of', len(context.controls))
print('gate:', narrow_report.decision.value, 'coverage:', narrow_report.requirement_coverage)
assert narrow_report.decision is GateDecision.PASS
assert narrow_report.requirement_coverage == 1.0
assert len(narrow_context.controls) < len(context.controls)

This is a critical failure pattern: evaluation is relative to its reference. If the reference omits privacy, platform, or domain controls, internal consistency can hide organizational nonconformance. Production systems need applicability resolution, provenance, freshness, and tests for context selection itself.

## Experiment 3 — Proportionate workflow routing

Full SDD is not the default for every change. This illustrative policy scores ambiguity, risk, blast radius, cross-repository impact, regulatory scope, and architectural change. It is inspectable and deterministic; your organization must calibrate dimensions and thresholds from its own outcomes.

In [ ]:
profiles = [
    ChangeProfile('copy edit', ambiguity=0, risk=0, blast_radius=1),
    ChangeProfile('known validation rule', ambiguity=1, risk=1, blast_radius=2),
    ChangeProfile('new shared API', ambiguity=3, risk=3, blast_radius=3, cross_repository=True),
    ChangeProfile('policy Q&A capability', ambiguity=4, risk=5, blast_radius=4, cross_repository=True, regulated=True, architecture_change=True),
]
for profile in profiles:
    recommendation = recommend_workflow(profile)
    print(f'{profile.name:25} score={recommendation.score:2}  {recommendation.workflow.value}')

assert recommend_workflow(profiles[0]).workflow is Workflow.DIRECT_CHANGE
assert recommend_workflow(profiles[1]).workflow is Workflow.LIGHTWEIGHT_SPEC
assert recommend_workflow(profiles[2]).workflow is Workflow.FULL_SDD
assert recommend_workflow(profiles[3]).workflow is Workflow.FULL_SDD_WITH_SPECIALIST_REVIEW

Change size is not the sole criterion. A one-line authorization change can warrant specialist review; a large generated test fixture can remain low risk. Routing should minimize total delay and risk across specification, execution, review, release, and operation—not maximize the number of SDD artifacts.

## Failure injection — lower-level policy override

Feature requirement `F-99` asks to send policyholder data to a public model API. Organization requirement `C-02` requires approved routes. The composer must preserve the company value, record both sources, and stop before execution.

In [ ]:
conflicting_context = compose_context(inject_feature_policy_override(requirements))
for conflict in conflicting_context.conflicts:
    print(conflict)
pii_control = next(c for c in conflicting_context.controls if c.control == 'pii_model_route')
assert pii_control.expected_value == 'approved_only'
assert conflicting_context.conflicts[0].winning_requirement_id == 'C-02'
assert conflicting_context.conflicts[0].rejected_requirement_id == 'F-99'

The mitigation is an explicit waiver workflow owned by the appropriate policy authority. A waiver needs scope, rationale, compensating controls, evidence, expiry, and revocation. The feature document cannot authorize itself.

## Experiment 4 — Change budget as a stop condition

Budgets make autonomy bounded and interruptible. Lower the allowed change from 20 to 10 files while keeping the approved design. The correct response is REVIEW: split the work or have an accountable reviewer change the budget. In our evaluator, boundary violations are fail-closed and therefore STOP.

In [ ]:
tight_boundary = replace(boundary, max_files_changed=10)
budget_report = evaluate_proposal(context, proposal, tight_boundary, human_approval_granted=True)
print(budget_report.decision.value, budget_report.boundary_violations)
assert budget_report.decision is GateDecision.STOP
assert any('change budget exceeded' in item for item in budget_report.boundary_violations)

## Evaluation summary

| Experiment | Observable result | What it teaches | What it does not prove |
| --- | --- | --- | --- |
| Prompt-only | STOP with missing, conflicting, overreach, scope and traceability findings | Fast execution can be organizationally wrong | A real model would make the fixture's exact choices |
| Layered proposal | REVIEW before approval; PASS after approval | Context, boundaries, evidence and approval are separate controls | The design or evidence is substantively correct |
| Context ablation | 100% coverage on a smaller reference | Clean metrics can hide missing governance context | More context is always better or should be loaded blindly |
| Proportionate routing | Four routes for four risk profiles | Ceremony should follow risk and uncertainty | The classroom thresholds fit your organization |
| Policy override | C-02 wins; F-99 is recorded as conflict | Lower layers cannot silently weaken higher policy | The exception is never justifiable |
| Tight budget | STOP at 14 proposed vs 10 allowed files | Agents need explicit stop conditions | File count alone measures risk |
| Repository candidates | Unsafe STOP; governed REVIEW; approved PASS | Actual code, independent checks, and approval are distinct | Local gates establish production behavior |

## Lab B — Apply the model to a repository change

The control simulation is necessary but not sufficient. Lab B starts with only JIRA-4821, discovers versioned requirements across organization/platform/domain/project/feature sources, evaluates applicability, copies a miniature repository into a temporary workspace, installs actual candidate source, runs unit tests plus independent policy/architecture checks, and writes an evidence bundle.

The unsafe candidate is an educational anti-pattern: it cannot call a provider or deploy anything, and its credential-like text is an inert sentinel. The governed candidate should STOP less, reach REVIEW when technical evidence passes, and reach PASS only when a training approval is explicitly supplied.

In [ ]:
from tempfile import TemporaryDirectory

repo_lab_candidates = [
    Path('repo_lab.py'),
    Path('curriculum/beginner/01-why-agentic-coding-changes-pdlc/repo_lab.py'),
]
repo_lab_path = next((path for path in repo_lab_candidates if path.exists()), None)
assert repo_lab_path is not None
repo_lab = runpy.run_path(str(repo_lab_path))
with TemporaryDirectory(prefix='course01-evidence-') as output:
    output_path = Path(output)
    unsafe_run = repo_lab['run_candidate']('unsafe', output_path)
    governed_run = repo_lab['run_candidate']('governed', output_path)
    approved_run = repo_lab['run_candidate']('governed', output_path, approve=True, label='governed-approved')
    evidence_files = sorted(path.name for path in (output_path / 'governed-approved').glob('*.json'))

print('unsafe:', unsafe_run['gate'], unsafe_run['checks'])
print('governed:', governed_run['gate'], governed_run['checks'])
print('approved:', approved_run['gate'], approved_run['checks'])
print('evidence:', evidence_files)
assert unsafe_run['gate'] == 'STOP'
assert governed_run['gate'] == 'REVIEW'
assert approved_run['gate'] == 'PASS'
assert len(evidence_files) == 10

The result separates technical conformance from approval authority. The unsafe implementation is caught by every check surface. The governed implementation passes specification, policy, architecture, tests, traceability, and independent review, but remains in REVIEW until named human roles approve. Even after PASS, the bundle lists tenant isolation, deployed residency, and representative answer quality as unverified production risks.

## Framework comparison: the primitive comes first

GitHub Spec Kit can package a constitution and phased quality gates; OpenSpec can package current specs and change deltas; Kiro can package requirements/design/tasks and route quick versus full flows; `AGENTS.md` can supply standing repository instructions. None replaces the primitive modeled here: determine authority, compose applicable requirements, detect conflict, bound execution, demand evidence, and route approval. Later courses will implement each framework rather than treating names as interchangeable.

## Production upgrade path

| Lab | Enterprise capability |
| --- | --- |
| Enum layers | Versioned registries with applicability, ownership and lifecycle |
| String controls | Typed contracts and policy-as-code where enforceable |
| In-memory conflict | Exception/waiver workflow with expiry and audit |
| Permission labels | Workload identity, scoped credentials, sandbox and egress policy |
| File budget | File/task/time/token/cost/retry/parallelism budgets and rollback |
| Requirement ID lists | Traceability graph across intent, ADRs, tasks, code, evidence, releases |
| Evidence presence | Independent evidence quality, provenance and reproducibility |
| Boolean approval | Named role, signed decision, separation of duties, revocation |
| Printed trace | Correlated, redacted, tamper-evident operational telemetry |

Production retries that can cause side effects must be bounded and idempotent. A failed or interrupted run must not deploy twice, duplicate migrations, or bypass an approval on retry.

## Exercises

1. Start from only JIRA-4821 and inventory the sources you need before implementation.
2. Remove `data_classification` from Lab B's change context and assert that privacy applicability becomes uncertain.
3. Add a 30-day retention control. Choose its layer, owner, applicability, and provenance.
4. Add a feature-level contradiction to Terraform requirement C-03 and test the conflict provenance.
5. Build a matrix showing which Lab B gate catches each unsafe candidate behavior and which risks remain uncovered.
6. Replace the classroom routing thresholds with a decision table for your organization.
7. Add a `max_tasks` budget and a failing test without weakening existing controls.
8. Identify one rule that belongs in `AGENTS.md`, one in CI policy, and one that requires runtime authorization.
9. Draw the owners and approvals for a real cross-repository change.

## Reflection and summary

Which decision in your current PDLC is usually discovered latest, and what durable artifact or gate could move it earlier without creating unnecessary ceremony?

Agentic coding changes the PDLC because execution can span more actions, files, tools, and time while organizational intent remains distributed. An enterprise Agentic PDLC makes applicable constraints durable, preserves decision authority, scopes execution, measures evidence, and learns from runtime behavior. The target is controlled iterative autonomy—not unlimited freedom and not waterfall with AI.